## INSERTAR INFORMACION EN SILVER Y GOLD

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalog_movie_project")
catalogo = dbutils.widgets.get("catalogo")

### Funciones para proceso de limpiza 

In [0]:
from pyspark.sql.functions import *

def limpiar_moneda(columna):
    return regexp_replace(col(columna), r"[^0-9]", "").cast("double")

In [0]:
df_moreinfo = spark.table(f"{catalogo}.bronze.moreinfo_pp")


### Limipieza campos tipo moneda para quedar numericos

In [0]:
df_clean_moreinfo = df_moreinfo \
    .withColumn("budget", limpiar_moneda("budget").cast("double")) \
    .withColumn("revenue", limpiar_moneda("revenue").cast("double")) \
    .select(
        "id",
        "runtime",
        "budget",
        "revenue",
        "poster_path",
        "backdrop_path"
    )

In [0]:
## spark.sql("delete from catalog_movie_project.silver.moreinfo_clean")

In [0]:
df_clean_moreinfo.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.silver.moreinfo_clean")

### Limpieza para tabla movies, para estandarizar generos

In [0]:
df_movies = spark.table(f"{catalogo}.bronze.movies")

In [0]:
df_clean_movies = df_movies.withColumn(
    "genres",
    array_distinct(
        transform(
            split(col("genres"), ","),
            lambda x: trim(lower(x))  # normaliza a minúsculas
                )
                )
    )\
    .select(
        "id"          
        ,"title"       
        ,"genres"      
        ,"language"    
        ,"user_score"  
        ,"runtime_hour"
        ,"runtime_min" 
        ,"release_date"
        ,"vote_count"  
      )

In [0]:
df_clean_movies.display()

In [0]:
df_clean_movies.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.silver.movies_clean")

In [0]:
%sql
select * from catalog_movie_project.bronze.moreinfo_pp

In [0]:
%sql
select * from catalog_movie_project.bronze.movies

In [0]:
%sql
select * from catalog_movie_project.silver.movies_clean

### Almacenamiento de informacion en Gold

In [0]:
df_film_detail = spark.table(f"{catalogo}.silver.filmdetails_clean")


In [0]:
df_master = df_clean_movies.alias("m").join(df_clean_moreinfo.alias("mi"), col("m.id") == col("mi.id"), "left").join(df_film_detail.alias("fd"),col("m.id")==col("fd.id"), "left")\
    .select(
        "m.id"
        ,"m.title"
        ,"m.genres"
        ,"m.language"
        ,"fd.director"
        ,"fd.top_billed"
        ,"m.runtime_min"
        ,col('m.release_date').alias('release_year')
        ,"m.user_score"
        ,"m.vote_count"
        ,"fd.budget_usd"
        ,"fd.revenue_usd"
        , (col("fd.revenue_usd") - col("fd.budget_usd")).alias("profit")
        ,(col("fd.revenue_usd") / col("fd.budget_usd") ).alias("roi") 
      )


In [0]:
df_master.display()

In [0]:
df_master.write \
    .mode("overwrite") \
    .saveAsTable(f"{catalogo}.golden.movie_master")

In [0]:
%sql
select * from catalog_movie_project.golden.movie_master